# OSSプロジェクトの健康診断

公開されている OSS リポジトリの開発履歴から、変更が集中しやすいファイル（ホットスポット）を予測する総合演習です。
リポジトリマイニング・前処理・ネットワーク分析・機械学習を1本のパイプラインにつなぎます。

- **Step 1** データ入手（PyDriller でコミット履歴を走査）
- **Step 2** 前処理（pandas でファイル単位の特徴量にする）
- **Step 3** ネットワーク分析（共変更ネットワークの中心性）
- **Step 4** 機械学習（ホットスポットを分類・評価）
- **Step 5** 可視化（分布とネットワーク）
- **Step 6** 考察（何が言えて、何に注意するか）

このノートブックは学習サイトのページ本文と同じコードです。Colab では上から順に実行できます。


まず必要なライブラリをインストールします（Colab では毎回必要です）。

In [ ]:
!pip install pydriller pandas networkx scikit-learn matplotlib

## Step 1. データ入手：コミット履歴を掘り起こす

PyDriller で公開リポジトリを走査し、変更ファイルごとに1行を作ります。
題材は小さめで実在する公開リポジトリ（ここでは PyDriller 自身の `ishepard/pydriller`）。
`url` を変えれば別のリポジトリで試せます。履歴取得はネットワーク接続が必要で、規模により数分かかります。


In [ ]:
from pydriller import Repository
import pandas as pd

url = "https://github.com/ishepard/pydriller"

# コミットを1件ずつたどり、変更ファイルごとに1行を作る
rows = []
for commit in Repository(url).traverse_commits():
    for mod in commit.modified_files:
        path = mod.new_path or mod.old_path
        if path is None:
            continue
        rows.append({
            "commit": commit.hash[:8],
            "author": commit.author.email,
            "date": commit.author_date,
            "file": path,
            "added": mod.added_lines,
            "deleted": mod.deleted_lines,
        })

df = pd.DataFrame(rows)
print(df.shape)
print(df.head())

## Step 2. 前処理：ファイル単位の特徴量にする

Python ファイル（`.py`）だけに絞り、パスが取れない変更を除きます。
各ファイルについて、変更回数（changes）・コードチャーン（churn）・関与作者数（authors）を作ります。


In [ ]:
# パスが取れない変更を除き、Pythonファイルに絞る
df = df[df["file"].notna()]
df = df[df["file"].str.endswith(".py")]

# コードチャーン = 追加行 + 削除行
df["churn"] = df["added"] + df["deleted"]

# ファイル単位に集計して特徴量表にする
feat = df.groupby("file").agg(
    changes=("commit", "nunique"),
    churn=("churn", "sum"),
    authors=("author", "nunique"),
).reset_index()

feat = feat.sort_values("changes", ascending=False)
print(feat.head())

## Step 3. ネットワーク分析：共変更の結び目を測る

同じコミットで一緒に変更されたファイル同士を辺で結び、共変更ネットワークを作ります。
次数中心性（多くのファイルと共に変わるほど高い）と媒介中心性（橋になっているほど高い）を測ります。


In [ ]:
import networkx as nx
from itertools import combinations

# 同じコミットで一緒に変わったファイルを辺で結ぶ
G = nx.Graph()
for commit, grp in df.groupby("commit"):
    files = sorted(grp["file"].unique())
    for a, b in combinations(files, 2):
        if G.has_edge(a, b):
            G[a][b]["weight"] += 1
        else:
            G.add_edge(a, b, weight=1)

# 次数中心性と媒介中心性
degree = nx.degree_centrality(G)
between = nx.betweenness_centrality(G)

cent = pd.DataFrame({
    "file": list(degree.keys()),
    "degree": list(degree.values()),
    "betweenness": [between[f] for f in degree],
})
print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())
print(cent.sort_values("betweenness", ascending=False).head())

## Step 4. 機械学習：ホットスポットを予測する

特徴量を1つの表にまとめ、変更回数が上位25%のファイルをホットスポット（正例）と定義します。
ラベルの定義に使った変更回数そのものは特徴量に入れず、残りの特徴量から予測できるかを試します。
`train_test_split` で分割し、`RandomForestClassifier` で分類、`classification_report` で評価します。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 中心性を特徴量表に結合（単独変更のファイルは中心性0で補完）
data = feat.merge(cent, on="file", how="left").fillna(0)

# ラベル: 変更回数が上位25%のファイルをホットスポット(1)
threshold = data["changes"].quantile(0.75)
data["hotspot"] = (data["changes"] >= threshold).astype(int)

# 変更回数そのものは特徴量に入れない（ラベルの定義に使ったため）
X = data[["churn", "authors", "degree", "betweenness"]]
y = data["hotspot"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=300, random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_report(y_test, pred))
# どの特徴量が効いたか
for name, imp in sorted(zip(X.columns, clf.feature_importances_),
                        key=lambda t: t[1], reverse=True):
    print(f"{name:12s} {imp:.3f}")

## Step 5. 可視化：分布とネットワークを眺める

変更回数の分布（ロングテールになりやすい）と、共変更ネットワークの中心部分を描きます。
図の見た目はレイアウトの乱数で変わりますが、構造の傾向は保たれます。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# 左: ファイルごとの変更回数の分布（ロングテール）
ax[0].hist(feat["changes"], bins=30, color="#0E6F5C")
ax[0].set_xlabel("changes per file")
ax[0].set_ylabel("number of files")
ax[0].set_title("Change frequency")

# 右: 共変更ネットワークの中心部分（次数の高い上位30ノード）
top_nodes = [n for n, _ in sorted(G.degree,
             key=lambda x: x[1], reverse=True)[:30]]
H = G.subgraph(top_nodes)
pos = nx.spring_layout(H, seed=42)
nx.draw_networkx_nodes(H, pos, ax=ax[1], node_size=120, node_color="#0E6F5C")
nx.draw_networkx_edges(H, pos, ax=ax[1], edge_color="#B7C0BB")
ax[1].set_title("Co-change network (top 30)")
ax[1].axis("off")

plt.tight_layout()
plt.show()

## Step 6. 考察：何が言えて、何に注意するか

少数のファイルに変更が集中し、それらが共変更ネットワークの結び目にもなっていること、
ホットスポットがチャーンと次数中心性からある程度予測できることが分かります。
ネットワーク由来の指標を足すと予測が補強される、というのがこの演習の要点です。

注意点も忘れずに。

- **相関であって因果ではない**：変更が多い＝悪いとは限らない（活発に育つ中心機能かもしれない）。
- **tangled commit**：無関係な変更が1コミットに混ざると、共変更が過大に見える。
- **リネーム・移動**：ファイル名の変更で履歴が途切れ、変更回数を過小評価することがある。

### 発展課題

- `url` を別の小さな公開リポジトリに変えて比べる。
- `Repository(url, since=..., to=...)` で期間を区切り、直近と全期間を比べる。
- 平均コミット間隔・直近変更からの経過日数などの特徴量を足す。
- コミットメッセージから `fix` / `bug` を拾い、修正が集中するファイルを調べる（SZZ 的な追跡へ）。
